# 04. 서류 투입 — 원본을 넣으면 갈라 보낸다

`서류투입` 폴더의 칸에 **원본**을 넣고 이 노트북을 돌리면 됩니다.

| 레인 | 대상 | 처리 |
|---|---|---|
| 압축분류 | 자재검수 · PHC_겉모양 · 시료채취 · 물시멘트비 · BSCW · JSP | 촬영정보·GPS 제거 + 1600px 압축 |
| 원본분류 | 의뢰시험_재하 · 의뢰시험_MT · 의뢰시험_일반 | **바이트 그대로** 보존 |
| 성적서 | 성적서_* · 송장 | 글자 추출 + 항목 자동 완성 |

> ★ **원본을 넣으세요.** 카카오톡으로 받은 사진은 이미 뭉개져 글자를 못 읽습니다.
> 프로그램은 원본에서 먼저 읽고(스캔) 그 다음에 압축합니다(실행).

In [ ]:
# 이 셀을 먼저 실행하세요. 어디서 열어도 프로젝트를 찾습니다.
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent                      # notebooks/ 에서 열었을 때
assert (ROOT / "main.py").exists(), f"프로젝트를 찾지 못했습니다: {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

def 표(머리, 행들):
    """리스트를 표로 보여 준다."""
    if not 행들:
        display(Markdown("_내용 없음_"))
        return
    md = "| " + " | ".join(str(h) for h in 머리) + " |\n"
    md += "|" + "|".join("---" for _ in 머리) + "|\n"
    for r in 행들:
        md += "| " + " | ".join("" if c is None else str(c) for c in r) + " |\n"
    display(Markdown(md))

from core.config import load_config

# 설정은 현재 폴더 -> 프로젝트 폴더 순으로 찾고, 없으면 예시에서 만들어 준다
후보 = [pathlib.Path.cwd() / "config.yaml", ROOT / "config.yaml"]
cfg = load_config(next((p for p in 후보 if p.exists()), None))
print("프로젝트:", ROOT)
print("설정 파일:", cfg.source)

## 1. 투입 폴더 확인

In [ ]:
from tasks.document_intake import DocumentIntake, Lane

intake = DocumentIntake(cfg)
print("투입 폴더:", intake.root, "— 있음" if intake.root.exists() else "— 없음")
표(["칸", "레인", "하위폴더 필요", "보낼 곳"],
   [(c.name, c.lane.value, "예" if c.require_sub else "", c.dest) for c in intake.categories])

### 하위폴더가 필요한 칸

폴더 이름이 곧 분류 기준입니다. **촬영일로는 맞출 수 없습니다.**

| 칸 | 폴더 이름 | 예 |
|---|---|---|
| BSCW · JSP | 타설일 | `0822` |
| 물시멘트비 | 시험 회차 | `01` |
| 의뢰시험_재하 | 시험일 동-번호 | `0824 113-152` |

## 2. 스캔 — 무엇이 어디로 갈지

**파일을 건드리지 않습니다.** 성적서는 이 단계에서 원본을 읽습니다.

In [ ]:
items, blocked = intake.scan()
print(len(items), "건 ·", len(blocked), "건 보류")

for lane in Lane:
    골라낸 = [i for i in items if i.lane is lane]
    if not 골라낸:
        continue
    display(Markdown(f"### {lane.value} — {len(골라낸)}건"))
    표(["칸", "파일", "읽어낸 내용", "보낼 곳"],
       [(i.category, i.src.name, i.요약, i.dest) for i in 골라낸])

## 3. 보류 · 경고

In [ ]:
for b in blocked:
    print(" -", b)
for i in items:
    for w in (i.parsed.경고 if i.parsed else []):
        print(f" ! {i.src.name}: {w}")
if not blocked and not any(i.parsed and i.parsed.경고 for i in items):
    print("이상 없음")

## 4. 실행

여기서부터 **파일이 실제로 움직입니다.**
압축분류는 원본을 `사진백업` 으로 옮기고 압축본을 목적지에 둡니다 (원본은 30일 보관).

아래 `실행하기 = True` 로 바꿔야 돌아갑니다.

In [ ]:
실행하기 = False       # <- True 로 바꾸면 실제로 옮깁니다

if 실행하기:
    결과 = intake.run(items, blocked)
    print(결과.요약)
    for p, e in 결과.failed:
        print(" [실패]", p.name, e)
else:
    결과 = intake.run(items, blocked, dry_run=True)
    print("미리보기:", len(결과.moved), "건이 옮겨질 예정")
    print("(실행하기 = False 라 실제로는 옮기지 않았습니다)")